# Tools and Functions

In [32]:
// IMPORTS

#include <iostream> 
#include <random>

In [33]:
// VALUE PRINT

template <typename T>
void vprint(const T& x) {
    std::cout << x << std::endl;
}

In [34]:
// ARRAY PRINT

template <typename T>
void aprint(const T* array, size_t size) {
    for (size_t i = 0; i < size; ++i) {
        std::cout << array[i] << " ";
    }
    std::cout << std::endl; 
}

In [35]:
// MATRIX PRINT (ONLY FOR INTEGER VALUES)

void mprint(int** matrix, size_t rows, size_t cols) {
    for (size_t i = 0; i < rows; ++i) {
        for (size_t j = 0; j < cols; ++j) {
            std::cout << matrix[i][j] << " ";
        }
        std::cout << std::endl;
    }
}

In [36]:
// RANDOM SETUP
// from 1 to 9

std::random_device rd; // Obtain a random number from hardware
std::mt19937 gen(rd()); // Seed the generator
std::uniform_int_distribution<> int_dis(1, 9);

In [37]:
// RANDOM GEN

int randInt(){
    return int_dis(gen);
}

# Cache Basics

Get some information about the CPU on the current system (in that case from the codespace).

In [38]:
! lscpu

Architecture:                       x86_64
CPU op-mode(s):                     32-bit, 64-bit
Byte Order:                         Little Endian
Address sizes:                      48 bits physical, 48 bits virtual
CPU(s):                             2
On-line CPU(s) list:                0,1
Thread(s) per core:                 2
Core(s) per socket:                 1
Socket(s):                          1
NUMA node(s):                       1
Vendor ID:                          AuthenticAMD
CPU family:                         25
Model:                              1
Model name:                         AMD EPYC 7763 64-Core Processor
Stepping:                           1
CPU MHz:                            3243.754
BogoMIPS:                           4890.84
Virtualization:                     AMD-V
Hypervisor vendor:                  Microsoft
Virtualization type:                full
L1d cache:                          32 KiB
L1i cache:                          32 KiB
L2 cache:           

Lets look at the cache by its own.

In [39]:
! lscpu | grep -i "cache"

L1d cache:                          32 KiB
L1i cache:                          32 KiB
L2 cache:                           512 KiB
L3 cache:                           32 MiB


We have 2 CPUs available with 32 KiB of L1 cache (sometimes also known as level 1 cache).

The L1 cache is a type of CPU cache that is closest to the processor cores. It is typically split into two distinct caches: L1i (L1 instruction cache) and L1d (L1 data cache). 

Purpose:

- L1i Cache (Instruction Cache):
    - Stores instructions that the CPU needs to execute.
    - Optimizes the fetching of instructions from memory to the CPU.
    - Improves the performance of the instruction pipeline by reducing the time it takes to fetch instructions.

- L1d Cache (Data Cache):
    - Stores data that the CPU needs to process.
    - Optimizes the fetching of data from memory to the CPU.
    - Enhances the performance of data retrieval operations.The L1 cache, or Level 1 cache, is a type of CPU cache that is closest to the processor cores. It is typically split into two distinct caches: L1i (L1 instruction cache) and L1d (L1 data cache). Here are the differences between them:

We also have 512 KiB of L2 and 32 MiB of L3 cache.

Let look at the cache line size.

In [40]:
! getconf LEVEL1_DCACHE_LINESIZE

64


In [41]:
! getconf LEVEL1_ICACHE_LINESIZE

64


In [42]:
! getconf LEVEL2_CACHE_LINESIZE

64


In [43]:
! getconf LEVEL3_CACHE_LINESIZE

64


The cache line size is 64 bytes.

This means, with a L1 cache size of 32 KiB and a cache line size of 64 bytes we have 500 cache lines.
When storing loading data in the L1 cache, we load a whole cache line, so in that case 64 bytes per load.

# Cpp Basics

Get the size of an integer in byte.

In [44]:
vprint(sizeof(int));

4


This means that we use 4 bytes (**32 bits**) per integer, so $−2^{31}$ to $231^{−1231}−1$ for signed integers.

## malloc

void* malloc(size_t size);

### How malloc Works

- Requesting Memory: When malloc is called, it requests a block of memory from the heap (**RAM**) of the specified size.
- Allocating Memory: The heap manager in the runtime library tries to find a suitable **contiguous** block of free memory that meets the requested size. If it finds such a block, it marks it as used and returns a **pointer** to the beginning of the block.
- Return Value: If successful, malloc returns a void* pointer to the beginning of the allocated memory block. If it fails (e.g., if there is not enough memory available), it returns NULL.

Lets make an array with 10 integer values.
(Note: We have to keep track of the size and / or type of the array to iterate over it.)

In [45]:
int* arr = (int*)malloc(10 * sizeof(int));

We allocate 10 * 4 bytes with malloc and cast the void pointer to an integer pointer that points to the first element.
This integer pointer must be stored in an integer pointer variable, in that case "arr". 

Lets set the value at positon [0] to 1.

In [46]:
arr[0] = 1;

And print it.

In [47]:
vprint(arr[0]);

1


Lets write a quick function to fill the array with random integer values.
(The array must be an integer array.)

In [48]:
void fillRandom(int* arr, size_t size) {

    for(int i = 0; i < size; i++){
        arr[i] = randInt();
    }

}

Lets fill the array with values.

In [49]:
fillRandom(arr, 10);

And print it.

In [50]:
aprint(arr, 10);

3 2 8 2 8 6 8 1 9 2 


## Vecotrs

Now lets make a function to calculate the dot product of 2 arrays.

In [51]:
int dot(int* arr1, int* arr2, size_t size){
    int sum = 0;
    for(int i = 0; i < size; i++){
        sum += arr1[i] * arr2[i];
    }
    return sum;
}

We know, that we have 64 bytes cache line size and 4 bytes per integer.
That means, that we can potentially load 16 integer in one cache line.
And that means, that we potentially only have to load 2 times from RAM to perform a 16 x 16 dot product.  

Note:
This is a simplified view, as real-world scenarios involve considerations like alignment and data structures etc.

In [52]:
int size = 16;

int* arr1 = (int*)malloc(size * sizeof(int));
int* arr2 = (int*)malloc(size * sizeof(int));

fillRandom(arr1, size);
fillRandom(arr2, size);

aprint(arr1, size);
aprint(arr2, size);

6 4 7 6 3 2 5 8 7 1 4 4 9 4 4 8 
5 2 5 9 9 2 3 4 2 8 3 5 4 3 7 7 


In [53]:
vprint(dot(arr1, arr2, size))

391


## Matrices

We want to create a matrix of size n x m or rows x cols.

In [54]:
size_t rows = 4;
size_t cols = 4; 

A matrix is a set of pointers that are pointing to set of values (vecotrs) each.
Thats why we need to create a set of integer pointers first. 

In [55]:
int** matrix = (int**)malloc(rows * sizeof(int*));

Lets fill our now pointers by allocating the space needed.

In [25]:
for (int i = 0; i < rows; ++i) {
    
    matrix[i] = (int*)malloc(cols * sizeof(int));

}

To fill it, we must create a new function.

(Note: We could use the fillRandom function too.)

In [26]:
void fillMatrixRandom(int** matrix, size_t rows, size_t cols){

    for(int i = 0; i < rows; i++){
        
        for(int j = 0; j < cols; j++){
            
            matrix[i][j] = randInt();

        }
    
    }

}

And fill our matrix.

In [27]:
fillMatrixRandom(matrix, rows, cols);

Now we can print it.

In [28]:
mprint(matrix, rows, cols);

6 9 6 2 
4 3 9 4 
6 2 3 1 
2 2 1 2 


At the end, we must free the allocated memory space.

In [29]:
void freeM(int** matrix, int n) {
    for (int i = 0; i < n; ++i) {
        free(matrix[i]); 
    }
    free(matrix); 
}

Lets clean everything up and make some functions. 

In [30]:
int* createVector(size_t size){
    int* v = (int*)malloc(size * sizeof(int));
    fillRandom(v, size);
    return v;
}

In [31]:
int** createMatrix(size_t rows, size_t cols){
    int** m = (int**)malloc(rows * sizeof(int*));
    for (int i = 0; i < rows; ++i) {
        matrix[i] = createVector(cols);
    }

    return m;
}

And finally, lets make the standart dot product for matrices.

(Note: For now, we assume that we have quadratic matrices, row = cols.)

In [32]:
void mdot(int** mat1, int** mat2, int** out, size_t size){

    for(int row = 0; row < size; row++){

        for(int col = 0; col < size; col++){

            for(int k = 0; k < size; k++){
                
                out[row][col] = mat1[row][k] * mat2[k][col];
            }
        }
    }
}

And test it.

In [56]:
size_t size = 4;

int** m1 = createMatrix(size, size);
int** m2 = createMatrix(size, size);
int** out = createMatrix(size, size);

mprint(m1, size, size);
// mprint(m2, size, size);

// mdot(m1, m2, out, size);

// mprint(out, size, size);

1 8 9 6 


In [ ]:
// free(arr);
// free(arr1);
// free(arr2);
// freeM(matrix, rows);